# Exp 3-v7 🚀 — TF-IDF + **핸드크래프트 피처 concat** (전처리 한계 돌파!)

## 왜 v6가 +0.06%p밖에 안 올랐나?
→ 전처리 개선만으로는 TF-IDF 한계 존재

## 🎯 진짜 전략: 핸드크래프트 피처 concat
TF-IDF가 **아예 못 보는** 신호를 직접 추가

| 피처 | negative | neutral | positive |
|------|----------|---------|----------|
| 느낌표 수 | 0.425 | 0.379 | **0.664** |
| 물음표 수 | 0.145 | **0.201** | 0.084 |
| ALL CAPS 단어 수 | **0.302** | 0.242 | 0.272 |
| 텍스트 길이 (단어 수) | 장문 neg 많음 | - | - |
| URL 존재 여부 | 0.023 | **0.047** | 0.037 |
| 이모티콘 존재 여부 | - | - | positive 신호 |

**input_size = 30000 (TF-IDF) + 6 (핸드크래프트) = 30006**

**예상 성능**: 68.36% → **70%+** 🎯

In [1]:
!pip install datasets scikit-learn -q

In [2]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from scipy.sparse import hstack, csr_matrix
import numpy as np, copy, re
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [3]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}|Dev:{len(dev_data)}|Test:{len(test_data)}|Classes:{output_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

Train:31232|Dev:5205|Test:5205|Classes:3


In [4]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.replace('`', "'")
    text = text.replace('****', ' bad ')
    text = text.replace('***', ' bad ')
    text = re.sub(r'!{3,}', ' verymuch ! ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'cannot', text)
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"'re", ' are', text)
    text = re.sub(r"'ve", ' have', text)
    text = re.sub(r"'ll", ' will', text)
    text = re.sub(r"'d", ' would', text)
    text = re.sub(r"'m", ' am', text)
    slangs = [
        (r'\bidk\b', 'i do not know'), (r'\bur\b', 'your'),
        (r'\bnaw\b', 'no'), (r'\bgonna\b', 'going to'),
        (r'\bwanna\b', 'want to'), (r'\blol\b', 'laughing'),
        (r'\bomg\b', 'oh my god'), (r'\bwtf\b', 'what the'),
        (r'\bugh\b', 'disgusting'), (r'\btho\b', 'though'),
        (r'\bkinda\b', 'kind of'), (r'\bcuz\b', 'because'),
        (r'\bsoo+\b', 'so'), (r'\bthx\b', 'thanks'),
        (r'\byep\b', 'yes'), (r'\byup\b', 'yes'),
        (r'\bnope\b', 'no'), (r'\btbh\b', 'to be honest'),
        (r'\bimo\b', 'in my opinion'),
    ]
    for pat, rep in slangs:
        text = re.sub(pat, rep, text)
    return text

def extract_handcraft(texts):
    """TF-IDF가 못 보는 신호 6개"""
    features = []
    for text in texts:
        t = str(text)
        tl = t.lower()
        words = t.split()
        feats = [
            min(t.count('!'), 5),                                          # 느낌표 수 (positive↑)
            min(t.count('?'), 5),                                          # 물음표 수 (neutral↑)
            sum(1 for w in words if w.isupper() and len(w) > 1),           # ALL CAPS 단어 수
            min(len(words), 50),                                           # 단어 수
            int(bool(re.search(r'http\S+', tl))),                          # URL 포함 여부
            int(any(e in tl for e in [':)', ':(', ':d', ':/', 'haha', 'hehe', 'lmao'])),  # 이모티콘
        ]
        features.append(feats)
    return np.array(features, dtype=np.float32)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=30000, preprocessor=preprocess_text, min_df=2)
vectorizer.fit(train_data['text'])

# 핸드크래프트 피처 확인
hc_train = extract_handcraft(train_data['text'])
print(f'핸드크래프트 피처 평균 (neg/neu/pos):')
feat_names = ['느낌표', '물음표', 'ALL_CAPS', '단어수', 'URL', '이모티콘']
labels_arr = np.array(train_labels)
for i, name in enumerate(feat_names):
    neg_m = hc_train[labels_arr==0, i].mean()
    neu_m = hc_train[labels_arr==1, i].mean()
    pos_m = hc_train[labels_arr==2, i].mean()
    print(f'  {name}: neg={neg_m:.3f} | neu={neu_m:.3f} | pos={pos_m:.3f}')

# TF-IDF + 핸드크래프트 concat
def build_features(data_split, fit=False):
    texts = data_split['text']
    tfidf_mat = vectorizer.transform(texts)
    hc_mat = csr_matrix(extract_handcraft(texts))
    combined = hstack([tfidf_mat, hc_mat])
    return torch.FloatTensor(combined.toarray()).to(device)

train_t = build_features(train_data)
dev_t   = build_features(dev_data)
test_t  = build_features(test_data)
dev_labels_t = torch.tensor(dev_data['label'], dtype=torch.long).to(device)
input_size = train_t.shape[1]
print(f'\n최종 입력 크기: {input_size} (TF-IDF 30000 + 핸드크래프트 6)')

핸드크래프트 피처 평균 (neg/neu/pos):
  느낌표: neg=0.385 | neu=0.362 | pos=0.635
  물음표: neg=0.142 | neu=0.180 | pos=0.084
  ALL_CAPS: neg=0.302 | neu=0.242 | pos=0.272
  단어수: neg=17.420 | neu=15.309 | pos=16.994
  URL: neg=0.023 | neu=0.047 | pos=0.037
  이모티콘: neg=0.043 | neu=0.078 | pos=0.081

최종 입력 크기: 11663 (TF-IDF 30000 + 핸드크래프트 6)


In [5]:
class MLP(nn.Module):
    def __init__(self, i, h, o, d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [6]:
# Exp3 Best Config
BEST_H,BEST_LR,BEST_D,BEST_WD,BEST_EP,BEST_BS=256,5.591937393848229e-05,0.3,0,50,256
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model=MLP(input_size,BEST_H,output_size,BEST_D).to(device)
opt=optim.Adam(model.parameters(),lr=BEST_LR,weight_decay=BEST_WD)
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
print('🚀 학습 시작...')
print('='*50)
for epoch in range(BEST_EP):
    model.train()
    for i in range(0,len(train_t),BEST_BS):
        bd=train_t[i:i+BEST_BS]
        bl=torch.tensor(train_labels[i:i+BEST_BS],device=device)
        loss=lfn(model(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(model.state_dict())
        print(f'✨ Epoch {epoch+1}/{BEST_EP}|Dev:{da:.4f}|NEW BEST!')
    elif (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{BEST_EP}|Dev:{da:.4f}')
print('='*50)
model.load_state_dict(best_state)
torch.save(best_state,'best_model_exp3_v7_handcraft.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장: best_model_exp3_v7_handcraft.pt')
print(f'📊 Dev: {best_dev:.4f} | Test: {test_acc*100:.2f}%')
print(f'\n🎯 목표 70%: {"달성! 🎉🎉🎉" if test_acc >= 0.70 else f"({test_acc*100:.2f}%, v6 대비 {(test_acc-0.6836)*100:+.2f}%p)"}')

🚀 학습 시작...
✨ Epoch 1/50|Dev:0.3817|NEW BEST!
✨ Epoch 2/50|Dev:0.4142|NEW BEST!
✨ Epoch 3/50|Dev:0.4430|NEW BEST!
✨ Epoch 4/50|Dev:0.4753|NEW BEST!
✨ Epoch 5/50|Dev:0.5028|NEW BEST!
✨ Epoch 6/50|Dev:0.5299|NEW BEST!
✨ Epoch 7/50|Dev:0.5522|NEW BEST!
✨ Epoch 8/50|Dev:0.5889|NEW BEST!
✨ Epoch 9/50|Dev:0.6184|NEW BEST!
✨ Epoch 10/50|Dev:0.6338|NEW BEST!
✨ Epoch 11/50|Dev:0.6467|NEW BEST!
✨ Epoch 12/50|Dev:0.6530|NEW BEST!
✨ Epoch 13/50|Dev:0.6557|NEW BEST!
✨ Epoch 14/50|Dev:0.6599|NEW BEST!
✨ Epoch 15/50|Dev:0.6621|NEW BEST!
✨ Epoch 16/50|Dev:0.6669|NEW BEST!
✨ Epoch 17/50|Dev:0.6690|NEW BEST!
✨ Epoch 18/50|Dev:0.6705|NEW BEST!
✨ Epoch 19/50|Dev:0.6742|NEW BEST!
✨ Epoch 20/50|Dev:0.6761|NEW BEST!
✨ Epoch 21/50|Dev:0.6765|NEW BEST!
✨ Epoch 22/50|Dev:0.6793|NEW BEST!
✨ Epoch 23/50|Dev:0.6817|NEW BEST!
✨ Epoch 24/50|Dev:0.6820|NEW BEST!
✨ Epoch 25/50|Dev:0.6841|NEW BEST!
✨ Epoch 26/50|Dev:0.6849|NEW BEST!
✨ Epoch 27/50|Dev:0.6882|NEW BEST!
✨ Epoch 30/50|Dev:0.6899|NEW BEST!
✨ Epoch 36/50|Dev:

In [7]:
from google.colab import files
files.download('best_model_exp3_v7_handcraft.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>